# Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

SEED = 42

# Load Data

In [2]:
df = pd.read_csv('cleaned_discrete_maths_exams.csv')
print(df.shape)
df.head(2)

(168, 18)


,paper,page,method,length,page_count,text,exam_date,semester_info,year,sitting,doc_type,text_clean,question,letter,roman,marks,total,topics
0,Discrete_Mathematics_2013_Final_E.pdf,1,text,740,12,1 WIT BACHELOR OF SCIENCE ...,DECEMBER,SEMESTER 1 - YEAR 1,2013,Final,Exam,1 wit bachelor of science (hons) in - applied ...,False,False,False,False,False,[]
1,Discrete_Mathematics_2013_Final_E.pdf,2,text,924,12,2 Question 1 (a) The universal set ...,DECEMBER,SEMESTER 1 - YEAR 1,2013,Final,Exam,"2 question 1 (a) the universal set is {1, 2, 3...",True,True,True,True,False,"['Propositional Logic', 'Relations']"


# Define Target and Features

In [3]:
target = 'sitting'  # Final vs Repeat

features_to_drop = ['paper', 'text', 'text_clean', 'topics', 'semester_info', target]
all_features = [c for c in df.columns if c not in features_to_drop]

cat_features = [c for c in all_features if df[c].dtype == 'object']
num_features = [c for c in all_features if df[c].dtype != 'object']

print(f"{cat_features = }")
print(f"{num_features = }")

cat_features = ['method', 'exam_date', 'doc_type']
num_features = ['page', 'length', 'page_count', 'year', 'question', 'letter', 'roman', 'marks', 'total']


# Train/Test Split

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    df[all_features], df[target],
    stratify=df[target], test_size=0.40, random_state=SEED
)
X_train.shape, X_test.shape

((100, 12), (68, 12))

# Encode Target

In [5]:
le = LabelEncoder()
le.fit(y_train)
y_train = le.transform(y_train)
y_test = le.transform(y_test)

print(le.classes_)  # shows which label is 0 and which is 1

['Final' 'Repeat']


# Encode Categorical Features

In [6]:
ohe = OneHotEncoder(sparse_output=False)
ohe.fit(X_train[cat_features])
X_train_cat = ohe.transform(X_train[cat_features])
X_test_cat = ohe.transform(X_test[cat_features])

# Scale Numeric Features

In [7]:
ss = StandardScaler()
ss.fit(X_train[num_features])
X_train_num = ss.transform(X_train[num_features])
X_test_num = ss.transform(X_test[num_features])

# Combine

In [8]:
X_train = np.hstack([X_train_cat, X_train_num])
X_test = np.hstack([X_test_cat, X_test_num])
X_train.shape, X_test.shape

((100, 15), (68, 15))

# Train & Compare Models

In [9]:
models = {
    'LR':             LogisticRegression(),
    'RR':             RidgeClassifier(),
    'DT':             DecisionTreeClassifier(),
    'DT(max_depth=5)': DecisionTreeClassifier(max_depth=5),
    'KNN':            KNeighborsClassifier(),
    'SVM':            SVC()
}

for name, model in models.items():
    model.fit(X_train, y_train)
    train_score = accuracy_score(y_train, model.predict(X_train))
    test_score  = accuracy_score(y_test,  model.predict(X_test))
    print(f"{name:20s}  train: {train_score:.2%}  test: {test_score:.2%}")

LR                    train: 88.00%  test: 82.35%
RR                    train: 86.00%  test: 80.88%
DT                    train: 100.00%  test: 100.00%
DT(max_depth=5)       train: 100.00%  test: 94.12%
KNN                   train: 89.00%  test: 80.88%
SVM                   train: 90.00%  test: 79.41%
